In [1]:
import sys
import pandas as pd

sys.path.append("../")

from src.agent import classify_intent
from sklearn.metrics import f1_score, classification_report

print("Evaluation setup loaded")


Evaluation setup loaded


In [3]:
golden_set = pd.read_csv(
    "../evaluation/golden_set_annotation.csv"
)

print("Golden Set:", len(golden_set))
print("Missing intents:", golden_set["intent"].isna().sum())

Golden Set: 200
Missing intents: 0


In [4]:
X_eval = golden_set["customer_message"]
y_eval = golden_set["intent"]

print("Final evaluation examples:", len(X_eval))

Final evaluation examples: 200


In [5]:
historical_pairs = pd.read_csv(
    "../data/processed/americanair_customer_pairs.csv"
)

golden_ids = set(golden_set["customer_tweet_id"])

train_data = historical_pairs[
    ~historical_pairs["customer_tweet_id"].isin(golden_ids)
].copy()

print("Historical training examples:", len(train_data))

Historical training examples: 36431


In [ ]:
final_llm_predictions = []

for message in X_eval:
    prediction = classify_intent(message)
    final_llm_predictions.append(prediction)

print("Finished:", len(final_llm_predictions))

In [ ]:
from sklearn.metrics import f1_score, classification_report

print("Final LLM Macro F1:",
      f1_score(y_eval, final_llm_predictions, average="macro"))

print("Final LLM Weighted F1:",
      f1_score(y_eval, final_llm_predictions, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_eval,
    final_llm_predictions,
    zero_division=0
))

In [9]:
final_results = pd.DataFrame({
    "model": ["GPT-5.4-mini"],
    "accuracy": [
        f1_score(y_eval, final_llm_predictions, average="weighted")
        # temporary; we'll calculate accuracy properly below
    ],
    "macro_f1": [
        f1_score(y_eval, final_llm_predictions, average="macro")
    ],
    "weighted_f1": [
        f1_score(y_eval, final_llm_predictions, average="weighted")
    ]
})

from sklearn.metrics import accuracy_score

final_results["accuracy"] = accuracy_score(
    y_eval,
    final_llm_predictions
)

final_results.to_csv(
    "../evaluation/final_llm_results.csv",
    index=False
)

final_results

,model,accuracy,macro_f1,weighted_f1
0,GPT-5.4-mini,0.635,0.628023,0.635873


In [10]:
evaluation_predictions = golden_set[
    ["customer_tweet_id", "customer_message", "intent"]
].copy()

evaluation_predictions["predicted_intent"] = final_llm_predictions

evaluation_predictions.to_csv(
    "../evaluation/evaluation_predictions.csv",
    index=False
)

print("Saved:", len(evaluation_predictions))

Saved: 200


In [11]:
misclassified = evaluation_predictions[
    evaluation_predictions["intent"]
    != evaluation_predictions["predicted_intent"]
].copy()

print("Misclassified:", len(misclassified))

misclassified[
    ["customer_message", "intent", "predicted_intent"]
].head(20)

Misclassified: 73


,customer_message,intent,predicted_intent
2,@AmericanAir Thanks! Will DM you,follow_up,non_support
5,@AmericanAir Of course no response. Because yo...,follow_up,non_support
7,@AmericanAir DMing now. Thanks.,follow_up,contact_support
9,@AmericanAir 3114,follow_up,other_unclear
10,@AmericanAir Possible cancellation???,flight_information,flight_disruption
14,@AmericanAir https://t.co/KSQSEKWRff,other_unclear,non_support
17,@AmericanAir WA 🌎,other_unclear,non_support
18,@AmericanAir Thanks - DM'ing now,follow_up,contact_support
19,@AmericanAir AA 2767 out of MIA to EWR.,follow_up,flight_information
21,@AmericanAir Flight 1904,follow_up,flight_information


In [12]:
confusion_pairs = (
    misclassified
    .groupby(["intent", "predicted_intent"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusion_pairs.head(15)

,intent,predicted_intent,count
28,follow_up,other_unclear,7
26,follow_up,non_support,6
24,follow_up,contact_support,6
19,flight_disruption,other_unclear,3
35,other_unclear,non_support,3
31,non_support,flight_disruption,3
30,loyalty_upgrade,non_support,3
13,contact_support,follow_up,3
17,flight_disruption,non_support,2
36,other_unclear,refund_compensation,2


In [13]:
followup_errors = misclassified[
    misclassified["intent"] == "follow_up"
]

followup_errors[
    ["customer_message", "intent", "predicted_intent"]
].head(20)

,customer_message,intent,predicted_intent
2,@AmericanAir Thanks! Will DM you,follow_up,non_support
5,@AmericanAir Of course no response. Because yo...,follow_up,non_support
7,@AmericanAir DMing now. Thanks.,follow_up,contact_support
9,@AmericanAir 3114,follow_up,other_unclear
18,@AmericanAir Thanks - DM'ing now,follow_up,contact_support
19,@AmericanAir AA 2767 out of MIA to EWR.,follow_up,flight_information
21,@AmericanAir Flight 1904,follow_up,flight_information
26,@AmericanAir DM sent.,follow_up,contact_support
27,@AmericanAir Appreciate what?,follow_up,other_unclear
29,@AmericanAir Correct.,follow_up,non_support


In [14]:
followup_confusions = (
    followup_errors["predicted_intent"]
    .value_counts()
)

followup_confusions

predicted_intent
other_unclear         7
non_support           6
contact_support       6
flight_information    2
onboard_aircraft      1
airport_gate_staff    1
Name: count, dtype: int64

In [15]:
disruption_errors = misclassified[
    (misclassified["intent"] == "flight_disruption") &
    (misclassified["predicted_intent"] == "other_unclear")
]

disruption_errors[
    ["customer_message", "intent", "predicted_intent"]
]

,customer_message,intent,predicted_intent
56,"@AmericanAir Yep, meant 5608. This is what 4.5...",flight_disruption,other_unclear
147,@AmericanAir So - not likely that AA would boo...,flight_disruption,other_unclear
193,@AmericanAir My husband works for Fox News thi...,flight_disruption,other_unclear


In [16]:
confusion_pairs.head(10)

,intent,predicted_intent,count
28,follow_up,other_unclear,7
26,follow_up,non_support,6
24,follow_up,contact_support,6
19,flight_disruption,other_unclear,3
35,other_unclear,non_support,3
31,non_support,flight_disruption,3
30,loyalty_upgrade,non_support,3
13,contact_support,follow_up,3
17,flight_disruption,non_support,2
36,other_unclear,refund_compensation,2


In [17]:
loyalty_errors = misclassified[
    (misclassified["intent"] == "loyalty_upgrade") &
    (misclassified["predicted_intent"] == "non_support")
]

loyalty_errors[
    ["customer_message", "intent", "predicted_intent"]
]

,customer_message,intent,predicted_intent
67,@AmericanAir Thanks! Doesn't look like I am ke...,loyalty_upgrade,non_support
131,@AmericanAir I’m not a million miler and am ju...,loyalty_upgrade,non_support
157,@AmericanAir When I fly @SouthwestAir I get fr...,loyalty_upgrade,non_support


In [18]:
unclear_errors = misclassified[
    (misclassified["intent"] == "other_unclear") &
    (misclassified["predicted_intent"] == "non_support")
]

unclear_errors[
    ["customer_message", "intent", "predicted_intent"]
]

,customer_message,intent,predicted_intent
14,@AmericanAir https://t.co/KSQSEKWRff,other_unclear,non_support
17,@AmericanAir WA 🌎,other_unclear,non_support
195,@AmericanAir No bueno. Wish u cared more about...,other_unclear,non_support


In [19]:
non_support_errors = misclassified[
    (misclassified["intent"] == "non_support") &
    (misclassified["predicted_intent"] == "flight_disruption")
]

non_support_errors[
    ["customer_message", "intent", "predicted_intent"]
]

,customer_message,intent,predicted_intent
51,@AmericanAir thanks for the quick response! Ho...,non_support,flight_disruption
77,@AmericanAir You don't need to check the statu...,non_support,flight_disruption
155,@AmericanAir @473126 Here is their secret -if ...,non_support,flight_disruption


In [20]:
confusion_pairs.to_csv(
    "../evaluation/confusion_pairs.csv",
    index=False
)

print("Saved confusion analysis")

Saved confusion analysis


In [21]:
summary = pd.DataFrame({
    "metric": ["Accuracy", "Macro F1", "Weighted F1"],
    "score": [
        accuracy_score(y_eval, final_llm_predictions),
        f1_score(y_eval, final_llm_predictions, average="macro"),
        f1_score(y_eval, final_llm_predictions, average="weighted")
    ]
})

summary

,metric,score
0,Accuracy,0.635000
1,Macro F1,0.628023
2,Weighted F1,0.635873


In [22]:
judge_sample = golden_set.sample(
    n=20,
    random_state=42
).copy()

print("Judge sample:", len(judge_sample))

Judge sample: 20


In [24]:
import src.agent as agent

In [25]:
print("Agent loaded")

Agent loaded


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retriever_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000
)

historical_vectors = retriever_vectorizer.fit_transform(
    historical_pairs["customer_message"]
)

print("Retriever loaded:", historical_vectors.shape)

Retriever loaded: (36531, 50000)


In [35]:
judge_replies = []

for message in judge_sample["customer_message"]:
    result = agent.run_agent(
        message,
        historical_pairs,
        retriever_vectorizer,
        historical_vectors
    )
    judge_replies.append(result["reply"])

print("Generated replies:", len(judge_replies))

Generated replies: 20


In [36]:
judge_data = judge_sample[
    ["customer_tweet_id", "customer_message", "intent"]
].copy()

judge_data["agent_reply"] = judge_replies

judge_data.head()

,customer_tweet_id,customer_message,intent,agent_reply
95,1460835,@AmericanAir @americanair - why not just tell ...,seat,Please send us your record locator so we can t...
15,1103882,@AmericanAir All is well! Fixed it with an agent,non_support,Awesome! We're glad to hear everything was sor...
30,1411117,@AmericanAir That doesn't seem right,follow_up,@Customer We can take a closer look for you. P...
158,2572457,@AmericanAir @729565 @329789 unfortunately the...,non_support,We’re sorry for the concern. We can’t assist w...
128,1084105,@AmericanAir Also our @46987 is flying @americ...,non_support,@375726 That’s just wonderful news! We can’t w...


In [37]:
def judge_reply(customer_message, agent_reply):
    prompt = f"""
You are evaluating an airline customer-support agent reply.

Customer message:
{customer_message}

Agent reply:
{agent_reply}

Score the reply from 1 to 5 on each criterion:

1. Relevance — does it address the customer's issue?
2. Helpfulness — does it provide a useful next step?
3. Grounding — does it avoid unsupported claims?
4. Clarity — is it concise and understandable?

Return ONLY this format:

relevance: X
helpfulness: X
grounding: X
clarity: X
"""
    
    response = agent.client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    
    return response.output_text.strip()

In [38]:
judge_result = judge_reply(
    judge_data.iloc[0]["customer_message"],
    judge_data.iloc[0]["agent_reply"]
)

print(judge_result)

relevance: 4
helpfulness: 4
grounding: 5
clarity: 5


In [39]:
judge_scores = []

for _, row in judge_data.iterrows():
    score = judge_reply(
        row["customer_message"],
        row["agent_reply"]
    )
    judge_scores.append(score)

print("Judged:", len(judge_scores))

Judged: 20


In [40]:
import re

def parse_judge_score(text):
    scores = {}
    
    for line in text.splitlines():
        match = re.match(r"(relevance|helpfulness|grounding|clarity):\s*(\d+)", line.strip())
        if match:
            scores[match.group(1)] = int(match.group(2))
    
    return scores


parsed_scores = [parse_judge_score(x) for x in judge_scores]

judge_data["relevance"] = [x.get("relevance") for x in parsed_scores]
judge_data["helpfulness"] = [x.get("helpfulness") for x in parsed_scores]
judge_data["grounding"] = [x.get("grounding") for x in parsed_scores]
judge_data["clarity"] = [x.get("clarity") for x in parsed_scores]

judge_data.head()

,customer_tweet_id,customer_message,intent,agent_reply,relevance,helpfulness,grounding,clarity
95,1460835,@AmericanAir @americanair - why not just tell ...,seat,Please send us your record locator so we can t...,4,4,5,5
15,1103882,@AmericanAir All is well! Fixed it with an agent,non_support,Awesome! We're glad to hear everything was sor...,5,2,5,5
30,1411117,@AmericanAir That doesn't seem right,follow_up,@Customer We can take a closer look for you. P...,3,4,5,5
158,2572457,@AmericanAir @729565 @329789 unfortunately the...,non_support,We’re sorry for the concern. We can’t assist w...,2,3,5,5
128,1084105,@AmericanAir Also our @46987 is flying @americ...,non_support,@375726 That’s just wonderful news! We can’t w...,2,1,3,4


In [41]:
judge_data[
    ["relevance", "helpfulness", "grounding", "clarity"]
].mean()

relevance      3.50
helpfulness    2.90
grounding      4.35
clarity        4.75
dtype: float64

In [42]:
judge_data.to_csv(
    "../evaluation/reply_quality_judge.csv",
    index=False
)

print("Saved reply quality evaluation:", len(judge_data))

Saved reply quality evaluation: 20


In [43]:
overall_reply_score = judge_data[
    ["relevance", "helpfulness", "grounding", "clarity"]
].mean().mean()

print("Overall Reply Quality:", round(overall_reply_score, 2), "/ 5")

Overall Reply Quality: 3.88 / 5


In [44]:
human_sample = golden_set.sample(
    n=30,
    random_state=123
).copy()

human_sample[
    ["customer_tweet_id", "customer_message"]
].to_csv(
    "../evaluation/human_agreement_sample.csv",
    index=False
)

print("Human agreement sample:", len(human_sample))

Human agreement sample: 30


In [46]:
human_labels = pd.read_csv(
    "../evaluation/human_agreement_independent.csv"
)

print("Human labels:", len(human_labels))
print("Missing labels:", human_labels["human_intent"].isna().sum())

Human labels: 30
Missing labels: 0


In [47]:
agreement = (
    human_labels["human_intent"].values
    == golden_set.set_index("customer_tweet_id")
        .loc[human_labels["customer_tweet_id"], "intent"]
        .values
)

print("Human agreement:", agreement.mean())
print("Agreement %:", round(agreement.mean() * 100, 1))

Human agreement: 0.7
Agreement %: 70.0


In [48]:
agreement_result = pd.DataFrame({
    "metric": ["Human agreement"],
    "score": [agreement.mean()]
})

agreement_result.to_csv(
    "../evaluation/human_agreement_results.csv",
    index=False
)

print("Saved human agreement result")

Saved human agreement result


In [49]:
escalation_results = []

for _, row in golden_set.iterrows():
    predicted_intent = final_llm_predictions[
        golden_set.index.get_loc(row.name)
    ]

    decision = agent.decide_escalation(predicted_intent)

    escalation_results.append({
        "customer_tweet_id": row["customer_tweet_id"],
        "intent": predicted_intent,
        "decision": decision["decision"],
        "reason": decision["reason"]
    })

escalation_results = pd.DataFrame(escalation_results)

print(escalation_results["decision"].value_counts())

decision
auto_handle    121
escalate        79
Name: count, dtype: int64


In [50]:
escalation_results.to_csv(
    "../evaluation/escalation_results.csv",
    index=False
)

print("Saved escalation evaluation:", len(escalation_results))

Saved escalation evaluation: 200


In [51]:
import os

print(os.listdir("../evaluation"))

['.DS_Store', 'escalation_results.csv', 'model_results.csv', 'final_llm_results.csv', 'human_agreement_results.csv', 'golden_set_annotation.csv', 'reply_quality_judge.csv', 'baseline_results.csv', 'evaluation_predictions.csv', 'human_agreement_independent.csv', 'confusion_pairs.csv']


In [52]:
final_summary = pd.DataFrame({
    "metric": [
        "Accuracy",
        "Macro F1",
        "Weighted F1",
        "Reply Quality",
        "Human Agreement"
    ],
    "score": [
        accuracy_score(y_eval, final_llm_predictions),
        f1_score(y_eval, final_llm_predictions, average="macro"),
        f1_score(y_eval, final_llm_predictions, average="weighted"),
        overall_reply_score,
        agreement.mean()
    ]
})

final_summary

,metric,score
0,Accuracy,0.635000
1,Macro F1,0.628023
2,Weighted F1,0.635873
3,Reply Quality,3.875000
4,Human Agreement,0.700000


In [53]:
final_summary.to_csv(
    "../evaluation/final_summary.csv",
    index=False
)

print("Saved final summary")

Saved final summary


In [2]:
reply_judge = pd.read_csv("../evaluation/reply_quality_judge.csv")

print("Rows:", len(reply_judge))
print("Columns:")
print(reply_judge.columns.tolist())

reply_judge.head()

Rows: 20
Columns:
['customer_tweet_id', 'customer_message', 'intent', 'agent_reply', 'relevance', 'helpfulness', 'grounding', 'clarity']


,customer_tweet_id,customer_message,intent,agent_reply,relevance,helpfulness,grounding,clarity
0,1460835,@AmericanAir @americanair - why not just tell ...,seat,Please send us your record locator so we can t...,4,4,5,5
1,1103882,@AmericanAir All is well! Fixed it with an agent,non_support,Awesome! We're glad to hear everything was sor...,5,2,5,5
2,1411117,@AmericanAir That doesn't seem right,follow_up,@Customer We can take a closer look for you. P...,3,4,5,5
3,2572457,@AmericanAir @729565 @329789 unfortunately the...,non_support,We’re sorry for the concern. We can’t assist w...,2,3,5,5
4,1084105,@AmericanAir Also our @46987 is flying @americ...,non_support,@375726 That’s just wonderful news! We can’t w...,2,1,3,4


In [3]:
human_reply_ratings = reply_judge[
    [
        "customer_tweet_id",
        "customer_message",
        "agent_reply"
    ]
].copy()

human_reply_ratings["human_relevance"] = ""
human_reply_ratings["human_helpfulness"] = ""
human_reply_ratings["human_grounding"] = ""
human_reply_ratings["human_clarity"] = ""

human_reply_ratings.to_csv(
    "../evaluation/human_reply_quality.csv",
    index=False
)

print("Saved human reply-rating file")

Saved human reply-rating file


In [4]:
import pandas as pd

judge = pd.read_csv("../evaluation/reply_quality_judge.csv")
evaluator = pd.read_csv("../evaluation/human_reply_quality.csv")

comparison = judge.merge(
    evaluator[
        [
            "customer_tweet_id",
            "human_relevance",
            "human_helpfulness",
            "human_grounding",
            "human_clarity"
        ]
    ],
    on="customer_tweet_id",
    how="inner"
)

criteria = [
    ("relevance", "human_relevance"),
    ("helpfulness", "human_helpfulness"),
    ("grounding", "human_grounding"),
    ("clarity", "human_clarity")
]

results = []

for criterion, evaluator_col in criteria:
    judge_scores = comparison[criterion]
    evaluator_scores = comparison[evaluator_col]

    exact_agreement = (
        judge_scores == evaluator_scores
    ).mean()

    mean_absolute_error = (
        (judge_scores - evaluator_scores).abs()
    ).mean()

    results.append({
        "criterion": criterion,
        "exact_agreement": exact_agreement,
        "mean_absolute_error": mean_absolute_error
    })

agreement_results = pd.DataFrame(results)

agreement_results

,criterion,exact_agreement,mean_absolute_error
0,relevance,0.40,0.90
1,helpfulness,0.30,0.90
2,grounding,0.45,0.75
3,clarity,0.70,0.30


In [5]:
print("Overall exact agreement:",
      round(
          (comparison["relevance"] == comparison["human_relevance"]).mean()
          + (comparison["helpfulness"] == comparison["human_helpfulness"]).mean()
          + (comparison["grounding"] == comparison["human_grounding"]).mean()
          + (comparison["clarity"] == comparison["human_clarity"]).mean()
      / 4, 3
      ))

Overall exact agreement: 1.325


In [6]:
all_exact = []

for criterion, evaluator_col in criteria:
    all_exact.extend(
        (comparison[criterion] == comparison[evaluator_col]).tolist()
    )

print("Overall exact agreement:", round(sum(all_exact) / len(all_exact), 3))

Overall exact agreement: 0.463


In [7]:
agreement_results.to_csv(
    "../evaluation/judge_evaluator_agreement.csv",
    index=False
)

print("Saved evaluator agreement results")

Saved evaluator agreement results


In [8]:
human_sample = reply_judge.sample(
    n=10,
    random_state=42
)[
    [
        "customer_tweet_id",
        "customer_message",
        "agent_reply"
    ]
].copy()

human_sample["human_relevance"] = ""
human_sample["human_helpfulness"] = ""
human_sample["human_grounding"] = ""
human_sample["human_clarity"] = ""

human_sample.to_csv(
    "../evaluation/human_reply_quality_10.csv",
    index=False
)

print("Saved 10-example human rating sample")

Saved 10-example human rating sample


In [9]:
# These are the 7 examples that were manually rated by the human.
human_ids = [
    1103882,
    294436,
    858641,
    1736540,
    2572457,
    1484538,
    701891
]

human_comparison = comparison[
    comparison["customer_tweet_id"].isin(human_ids)
].copy()

print("Human-rated examples:", len(human_comparison))

human_agreement_results = []

for criterion, human_col in criteria:
    judge_scores = human_comparison[criterion]
    human_scores = human_comparison[human_col]

    exact_agreement = (
        judge_scores == human_scores
    ).mean()

    mean_absolute_error = (
        (judge_scores - human_scores).abs()
    ).mean()

    human_agreement_results.append({
        "criterion": criterion,
        "exact_agreement": exact_agreement,
        "mean_absolute_error": mean_absolute_error
    })

human_agreement_results = pd.DataFrame(
    human_agreement_results
)

human_agreement_results

Human-rated examples: 7


,criterion,exact_agreement,mean_absolute_error
0,relevance,0.428571,1.000000
1,helpfulness,0.142857,1.142857
2,grounding,0.571429,0.857143
3,clarity,0.714286,0.285714


In [10]:
all_human_exact = []

for criterion, human_col in criteria:
    all_human_exact.extend(
        (
            human_comparison[criterion]
            == human_comparison[human_col]
        ).tolist()
    )

overall_human_agreement = (
    sum(all_human_exact) / len(all_human_exact)
)

print(
    "Overall human-LLM judge exact agreement:",
    round(overall_human_agreement, 3)
)

print(
    "Percentage:",
    round(overall_human_agreement * 100, 1),
    "%"
)

Overall human-LLM judge exact agreement: 0.464
Percentage: 46.4 %


In [11]:
human_agreement_results["sample_size"] = len(human_comparison)

human_agreement_results.to_csv(
    "../evaluation/human_agreement_judge.csv",
    index=False
)

print("Saved human-LLM judge agreement results")

Saved human-LLM judge agreement results


In [12]:
import pandas as pd

predictions = pd.read_csv(
    "../evaluation/evaluation_predictions.csv"
)

print("Rows:", len(predictions))
print("Columns:")
print(predictions.columns.tolist())

predictions.head()

Rows: 200
Columns:
['customer_tweet_id', 'customer_message', 'intent', 'predicted_intent']


,customer_tweet_id,customer_message,intent,predicted_intent
0,1952986,@AmericanAir can't wait to switch to a @Delta ...,non_support,non_support
1,548815,@AmericanAir Must be a long walk.........,non_support,non_support
2,2308158,@AmericanAir Thanks! Will DM you,follow_up,non_support
3,1528201,@AmericanAir ALready did and no help. very dec...,follow_up,follow_up
4,2813114,@AmericanAir Thank you!! Hoping it works out.,non_support,non_support


In [13]:
misclassified = predictions[
    predictions["intent"] != predictions["predicted_intent"]
].copy()

confusion_pairs = (
    misclassified
    .groupby(["intent", "predicted_intent"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusion_pairs

,intent,predicted_intent,count
28,follow_up,other_unclear,7
26,follow_up,non_support,6
24,follow_up,contact_support,6
19,flight_disruption,other_unclear,3
35,other_unclear,non_support,3
31,non_support,flight_disruption,3
30,loyalty_upgrade,non_support,3
13,contact_support,follow_up,3
17,flight_disruption,non_support,2
36,other_unclear,refund_compensation,2


In [14]:
failure_cases = {
    "Context-dependent follow-ups": (
        "follow_up",
        ["other_unclear", "non_support", "contact_support"]
    ),
    "Ambiguous flight disruptions": (
        "flight_disruption",
        ["other_unclear"]
    ),
    "Loyalty/benefit ambiguity": (
        "loyalty_upgrade",
        ["non_support"]
    ),
    "Very low-information messages": (
        "other_unclear",
        ["non_support"]
    ),
    "Flight-related non-support chatter": (
        "non_support",
        ["flight_disruption"]
    )
}

for failure_name, (true_intent, predicted_intents) in failure_cases.items():
    print("\n" + "=" * 70)
    print(failure_name)
    print("=" * 70)

    cases = misclassified[
        (misclassified["intent"] == true_intent) &
        (misclassified["predicted_intent"].isin(predicted_intents))
    ].head(2)

    for _, row in cases.iterrows():
        print("\nTweet ID:", row["customer_tweet_id"])
        print("Customer:", row["customer_message"])
        print("Actual:", row["intent"])
        print("Predicted:", row["predicted_intent"])


Context-dependent follow-ups

Tweet ID: 2308158
Customer: @AmericanAir Thanks! Will DM you
Actual: follow_up
Predicted: non_support

Tweet ID: 1075780
Customer: @AmericanAir Of course no response. Because you don’t care
Actual: follow_up
Predicted: non_support

Ambiguous flight disruptions

Tweet ID: 701891
Customer: @AmericanAir Yep, meant 5608. This is what 4.5 hours in an airport on Thanksgiving does to a person.
Actual: flight_disruption
Predicted: other_unclear

Tweet ID: 162340
Customer: @AmericanAir So - not likely that AA would book a pax with that tight a connection, but the replacement F/A is ok?
Actual: flight_disruption
Predicted: other_unclear

Loyalty/benefit ambiguity

Tweet ID: 904832
Customer: @AmericanAir Thanks! Doesn't look like I am keeping that status this year :(
Actual: loyalty_upgrade
Predicted: non_support

Tweet ID: 2119599
Customer: @AmericanAir I’m not a million miler and am just a small fish in the frequent flier pool, but I️ do try to stay loyal to AA...